In [341]:
%load_ext autoreload
%autoreload 2
import sys
import pandas as pd
from pathlib import Path
import numpy as np

sys.path.insert(0, str(Path().resolve().parents[0]))
from transform.clean import normalize_paint, merge_variants, classify_exclusives, check_category_for_exclusivity, check_grade, parse_price, parse_year, to_list

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [342]:
INFOBOX_FIELDS = ["kit_name","image","categories","franchise","run","release date", "materials", "scale", "classification", 
                  "image_url", "price", "need glue?", "japanese name", "model of", "jan/isbn", "lineup no.", "variant of", "subtitle", "need to paint", "illustration by",
                  "need paint?","exclusive to", "imgsize"]

PAINT_COLS = ["need paint", "need paint?", "need to paint?"]

In [343]:
# Acquire path for dataset (it's in a folder on an upper level)
parent_path = Path().resolve().parents[0]
json_file = parent_path / "data/final/bandai_dataset_2026-07-01.jsonl"


In [344]:
#Create df
df = pd.read_json(json_file, lines=True)

In [345]:
# Pull out image URL into separate column and drop original image column
df['image_url'] = df['image'].map(lambda x: x.get('url', x) if isinstance(x, dict) else None)
df = df.drop(columns=['image'])

In [346]:
# Pull out all infobox fields into their own columns
df = pd.concat([df.drop(['infobox'], axis=1), df['infobox'].apply(pd.Series)], axis=1)
df = df.replace(r'^\s*$', np.nan, regex=True)

In [347]:
df.columns

Index(['kit_name', 'categories', 'image_url', 'image', 'classification',
       'lineup no.', 'scale', 'franchise', 'release date', 'price', 'jan/isbn',
       'need glue?', 'materials', 'run', 'model of', 'name', 'japanese name',
       'need paint?', 'exclusive to', 'variant of', 'subtitle',
       'need to paint?', 'availability', 'for use with', 'imgsize',
       'illustration by', 'sculptor', '1', 'add-on for', 'variant',
       'character design', 'figure sculpt', 'illustration', '2', 'need paint',
       'cg works by', 'finish work by'],
      dtype='str')

In [348]:
# Check to see if rows have more than 1 paint column
conflict = df[PAINT_COLS].notna().sum(axis=1) > 1
df = df.drop(columns=['need paint']) # 'need paint' only has 1 row with this not equal to NaN, and it's not even visible on the wiki page :D
df['need_paint_clean'] = df['need paint?'].apply(normalize_paint)
df['need_to_paint_clean'] = df['need to paint?'].apply(normalize_paint)

real_conflict = (df['need_paint_clean'].notna() & df['need_to_paint_clean'].notna() & (df['need_paint_clean'] != df['need_to_paint_clean']))
print(df.loc[real_conflict, ['kit_name', 'need paint?', 'need to paint?']])

df['need_paint'] = df['need_paint_clean'].combine_first(df['need_to_paint_clean'])
df = df.drop(columns=['need paint?', 'need to paint?', 'need_paint_clean', 'need_to_paint_clean'])

                                                              kit_name  \
1571  HGUC RX-0 Unicorn Gundam (Destroy Mode) (Painting Model Edition)   
1965                                  MG RX-78-3 G-3 Gundam (Ver. 2.0)   

     need paint?   need to paint?  
1571    Optional              Yes  
1965          No  Yes (figurines)  


In [349]:
from difflib import get_close_matches
cols = list(df.columns)
for c in cols:
    matches = get_close_matches(c, cols, n=3, cutoff=0.8)
    if len(matches) > 1:
        print(c, "~", matches)

variant of ~ ['variant of', 'variant']
illustration by ~ ['illustration by', 'illustration']
variant ~ ['variant', 'variant of']
illustration ~ ['illustration', 'illustration by']


In [350]:
# Compare variant columns and combine into one
VARIANT_COL = [c for c in df.columns if 'variant' in c.lower()]
variant_conflict = df[VARIANT_COL].notna().sum(axis=1) > 1
print(f"{variant_conflict.sum()} rows with both variant columns populated")
df.loc[variant_conflict, ['kit_name','variant']]

df['variant_of'] = df.apply(merge_variants, axis=1)
df = df.drop(columns=VARIANT_COL)

1 rows with both variant columns populated


In [351]:
# These really don't relate to the gunpla itself but rather the illustrator of the gundam or for figures + plus the random "1", "2" columns
df = df.drop(columns=['illustration by', 'image', 'sculptor','1','2','illustration','cg works by', 'finish work by','imgsize','figure sculpt','character design'])

In [352]:
df['exclusivity_type'] = df['exclusive to'].apply(classify_exclusives)  
print(df['exclusivity_type'].value_counts(dropna=False))
print(df.loc[df['exclusivity_type'] == 'Other', 'kit_name'].unique())

exclusivity_type
NaN           2199
Storefront      80
Event           28
Other            6
Magazine         3
Lottery          3
Campaign         2
Name: count, dtype: int64
<ArrowStringArray>
['1/100 MBF-P03secondL Gundam Astray Blue Frame Second L (Red Version)',
                              'HGPR Bracer Phoenix Set (Metallic Ver.)',
                          'HGPR Gipsy Avenger (DX Set) (Metallic Ver.)',
                               'HGPR Obsidian Fury Set (Metallic Ver.)',
       'MG MBF-P02 Gundam Astray Red Frame (Plating Frame/Clear Color)',
                               'PG RX-78-2 Gundam (Chrome Plated Ver.)']
Length: 6, dtype: str


In [353]:
# Create is_exclusive column
df['is_exclusive'] = df['categories'].apply(lambda cats: isinstance(cats, list) and any('exclusive' in c.lower() for c in cats))
print(f'{df['is_exclusive'].sum()} Exclusive kits out of {len(df)} kits')

needs_backfill = df['exclusive to'].isna() & df['is_exclusive']
inferred = df.loc[needs_backfill, 'categories'].apply(lambda cats: pd.Series(check_category_for_exclusivity(cats), index=['exclusivity_type', 'matched_category']))

df.loc[needs_backfill, 'exclusive_channel_type_from_category'] = inferred['exclusivity_type']
df.loc[needs_backfill, 'exclusive_value_from_category'] = inferred['matched_category']

print(f"Backfilled {inferred['exclusivity_type'].notna().sum()} of {needs_backfill.sum()} exclusive-but-unlabeled rows")
print(df.loc[needs_backfill & inferred['exclusivity_type'].notna(), ['kit_name', 'categories', 'exclusive_channel_type_from_category']].head(20))

518 Exclusive kits out of 2321 kits
Backfilled 352 of 408 exclusive-but-unlabeled rows
                                                                                                                        kit_name  \
68                                                                                                         1/100 RX-78F00 Gundam   
72                                                                 1/100 ZGMF-1000 ZAKU Warrior (Live Concert Extra Finish Ver.)   
78                                                                                1/100 ZGMF-X10A Freedom Gundam (Deactive Mode)   
94                                                           1/144 Destiny Gundam & Strike Freedom Gundam (Clear Color Ver. Set)   
106                                                           1/144 GAT-X105+AQM/E-X03 Launcher Strike Gundam (Clear Color Ver.)   
116                                                                               1/144 GAT-X303 Aegis Gundam (Clear Colo

In [354]:
pd.set_option('display.max_rows', 21)
still_unmatched = needs_backfill & inferred['exclusivity_type'].isna()
print(f"{still_unmatched.sum()} exclusive rows still unclassified")

unmatched_cats = df.loc[still_unmatched, 'categories'].explode()
unmatched_cats[unmatched_cats.str.contains('exclusive', case=False, na=False)].value_counts().head(30)

# We can probably leave the remaining exclusive kits that don't have a specification alone, stuff like CD-exclusive/ molds/ aren't specific enough

56 exclusive rows still unclassified


categories
Exclusives                    53
Event exclusives              16
CD-exclusive items             2
Exclusive-only molds           2
Exclusives／Regular Release     2
Collaboration exclusives       1
Name: count, dtype: int64

In [355]:
ADDON_COLS = [c for c in df.columns if c in ['for use with', 'add-on for']]
conflict = df[ADDON_COLS].notna().sum(axis=1) > 1
print(f"{conflict.sum()} rows with both columns populated")
df.loc[conflict, ['kit_name'] + ADDON_COLS]

13 rows with both columns populated


,kit_name,for use with,add-on for
240,1/144 c.o.v.e.r.-kit-4 for HGUC Zudah,HGUC EMS-10 Zudah,HGUC EMS-10 Zudah
241,1/144 c.o.v.e.r.-kit 5 for HGUC Powered GM,HGUC RGM-79 Powered GM,HGUC RGM-79 Powered GM
242,1/144 c.o.v.e.r kit 3 for HGUC Gundam GP-01,"HGUC RX-78GP01 Gundam ""Zephyranthes""","HGUC RX-78GP01 Gundam ""Zephyranthes"""
584,Blue Destiny Conversion Parts for MG RX-79(G) Gundam,MG RX-79［G］ Gundam Ground Type,1/100 MG RX-79[G] Gundam Ground Type
666,Full Armor Gundam Conversion Kit for HGUC Gundam,HGUC RX-78-2 Gundam (2001),HGUC RX-78-2 Gundam (2001)
674,GM Command Conversion Parts for MG GM Type C,MG RGM-79C GM Type C (Earth Type),MG RGM-79C GM Type C Earth/Space Type
1722,High Detail Manipulator 154 (Colored) for 1/144 Gundam Exia 1,HG00 GN-001 Gundam Exia,HG00 GN-001 Gundam Exia
1723,High Detail Manipulator 40 For 1/144 Sword Strike Gundam,HGGS GAT-X105+AQM/E-X01 Aile Strike Gundam,HGGS GAT-X105+AQM/E-X01 Aile Strike Gundam + 1/144 GAT-X105+AQM/E-X02 Sword Strike Gundam (Parts)
2017,MG Zaku F2 (Anime Ver.) Conversion Parts,MG MS-06F2 Zaku II F2 Type (Zeon Ver.),MG MS-06F2 Zaku II F2 Type (Zeon/E.F.S.F.)
2036,Musha Gundam Conversion kit for HGUC Gundam Mk-II,HGUC RX-178 Gundam Mk-II (Titans),HGUC RX-178 Gundam Mk-II (Titans) (or any of it's variants)


In [356]:
# Drop add-on for as it's either a duplicate of for use with, or it's not a searchable kit
df = df.rename(columns={'for use with' : 'used_for'})
df = df.drop(columns=['add-on for', 'name'])
df.loc[df['used_for'].notna(), ['kit_name','used_for']]

,kit_name,used_for
70,1/100 Shining Gundam Head Parts (Anime Ver.) for MG Shining Gundam,MG GF13-017NJ Shining Gundam
99,1/144 Fighting Action Body (Beam Rifle) for FG RX-78-2 Gundam,FG RX-78-2 Gundam
126,"1/144 GM Type C ""Wagtail"" Conversion Parts",GUC RGM-79C GM Type C
132,1/144 Gundam Astraea Conversion Kit,HG00 GN-001 Gundam Exia
133,1/144 Gundam Astraea Parts for RG Gundam Exia,RG GN-001 Gundam Exia
...,...,...
2286,Gundam Color Hyaku Shiki Gold,HGUC MSN-00100 Hyaku Shiki (Revive Ver.)
2287,Gundam Color Set for HGUC Zaku II,HGUC MS-06F Zaku II
2288,Gundam Color for 1/144 Strike Freedom Gundam (Deactive Mode),1/144 ZGMF-X20A Strike Freedom Gundam
2289,Gundam Color for HG 00 Gundam,HG00 GN-0000 00 Gundam


In [357]:
df.loc[df['exclusive_channel_type_from_category'].notna(), ['exclusive_channel_type_from_category','exclusive_value_from_category']]

,exclusive_channel_type_from_category,exclusive_value_from_category
68,Storefront,Gundam Factory Yokohama
72,Event,Chara Hobby 2007 exclusives
78,Event,Plamodel Radicon Show exclusives
94,Event,Bandai Museum exclusives
106,Campaign,Campaign exclusives
...,...,...
2249,Event,SD Gundam World Sangoku Soketsuden
2282,Storefront,Bandai Hobby Online Shop exclusives
2296,Storefront,Bandai Hobby Online Shop exclusives
2310,Storefront,Bandai Hobby Online Shop exclusives


In [358]:
df.columns

Index(['kit_name', 'categories', 'image_url', 'classification', 'lineup no.',
       'scale', 'franchise', 'release date', 'price', 'jan/isbn', 'need glue?',
       'materials', 'run', 'model of', 'japanese name', 'exclusive to',
       'subtitle', 'availability', 'used_for', 'need_paint', 'variant_of',
       'exclusivity_type', 'is_exclusive',
       'exclusive_channel_type_from_category',
       'exclusive_value_from_category'],
      dtype='str')

In [359]:
pd.set_option('display.max_rows', 25)

df.notna().mean().sort_values(ascending=False)

kit_name                                1.000000
categories                              1.000000
is_exclusive                            1.000000
variant_of                              1.000000
franchise                               0.993537
run                                     0.992676
release date                            0.990090
materials                               0.985351
scale                                   0.981904
classification                          0.977596
image_url                               0.973287
price                                   0.970702
need glue?                              0.969410
japanese name                           0.869453
model of                                0.809134
jan/isbn                                0.797932
lineup no.                              0.597156
subtitle                                0.321413
need_paint                              0.188281
exclusive_value_from_category           0.151659
exclusive_channel_ty

In [360]:
df = df.rename(columns={'release date': 'release_date', 'need glue?' : 'glue_needed', 'lineup no.':'lineup_num', 'model of':'model_of', 'exclusive to':'exclusive_to', 'japanese name':'japanese_name'})

In [361]:
df.columns

Index(['kit_name', 'categories', 'image_url', 'classification', 'lineup_num',
       'scale', 'franchise', 'release_date', 'price', 'jan/isbn',
       'glue_needed', 'materials', 'run', 'model_of', 'japanese_name',
       'exclusive_to', 'subtitle', 'availability', 'used_for', 'need_paint',
       'variant_of', 'exclusivity_type', 'is_exclusive',
       'exclusive_channel_type_from_category',
       'exclusive_value_from_category'],
      dtype='str')

In [362]:
print(df['used_for'].dropna().head(10))

70                              MG GF13-017NJ Shining Gundam
99                                         FG RX-78-2 Gundam
126                                    GUC RGM-79C GM Type C
132                                  HG00 GN-001 Gundam Exia
133                                    RG GN-001 Gundam Exia
201                            HGGS GAT-X131 Calamity Gundam
207               HGGS GAT-X105+AQM/E-X01 Aile Strike Gundam
208               HGGS GAT-X105+AQM/E-X01 Aile Strike Gundam
209                    HGGS ZGMF-X56S/α Force Impulse Gundam
210    HGGS Force Impulse Gundam, 1/144 Blast Impulse Gundam
Name: used_for, dtype: str


In [363]:
df.columns.tolist()

['kit_name',
 'categories',
 'image_url',
 'classification',
 'lineup_num',
 'scale',
 'franchise',
 'release_date',
 'price',
 'jan/isbn',
 'glue_needed',
 'materials',
 'run',
 'model_of',
 'japanese_name',
 'exclusive_to',
 'subtitle',
 'availability',
 'used_for',
 'need_paint',
 'variant_of',
 'exclusivity_type',
 'is_exclusive',
 'exclusive_channel_type_from_category',
 'exclusive_value_from_category']

In [364]:
df = df.drop(columns=['exclusivity_type'])
df = df.rename(columns={'used_for' : 'requires_kit'})

In [365]:
df['exclusive_channel_type'] = df['exclusive_to'].apply(classify_exclusives).combine_first(df['exclusive_channel_type_from_category'])

In [366]:
df = df.drop(columns=['exclusive_channel_type_from_category'])

In [367]:
df.notna().mean().sort_values(ascending=False)

kit_name                         1.000000
categories                       1.000000
is_exclusive                     1.000000
variant_of                       1.000000
franchise                        0.993537
run                              0.992676
release_date                     0.990090
materials                        0.985351
scale                            0.981904
classification                   0.977596
image_url                        0.973287
price                            0.970702
glue_needed                      0.969410
japanese_name                    0.869453
model_of                         0.809134
jan/isbn                         0.797932
lineup_num                       0.597156
subtitle                         0.321413
exclusive_channel_type           0.204222
need_paint                       0.188281
exclusive_value_from_category    0.151659
exclusive_to                     0.052994
requires_kit                     0.015511
availability                     0

In [368]:
df['grade'] = df.apply(lambda r: check_grade(r['kit_name'], r['classification']), axis=1)
df[['price_value', 'price_currency']] = df['price'].apply(lambda x: pd.Series(parse_price(x)))
pd.set_option('display.max_colwidth', None)
print(df.loc[[185, 213, 673, 678, 682, 683, 875, 1778, 2066, 2067, 2194, 2252, 2253, 2300], ['kit_name', 'price']].apply(
    lambda r: (r['kit_name'], parse_price(r['price'])), axis=1
))
df = df.drop(columns=['price'])
df['release_year'] = df['release_date'].apply(parse_year)

185                           (1/144 RX-78-2 Gundam (Revival Ver.) (GUNPLA POP-UP RUNNER'S GATE), (550, JPY))
213                                                             (1/144 XXXG-01H Gundam Heavyarms, (500, JPY))
673                                         (Full Mechanics ZGMF-X10A Freedom Gundam (Ver. GCP), (5600, JPY))
678                                                    (Gundam Assemble ST01 Heroic Beginnings, (34.99, USD))
682                                               (Gundarium Alloy Model 1/144 RX-78-2 Gundam, (220000, JPY))
683                                                                   (Gunpla 40th Memorial Set, (4546, JPY))
875                                                    (HGBD PEN-01M Momokapool (Ver. Zaishin), (150.0, USD))
1778            (MG GBK-20 Gundam Astray (The Gundam Base Korea 20th Anniversary Memorial Ver.), (8000, JPY))
2066               (Premium Card Collection Gundam Assemble Set -Mobile Suit Gundam GQuuuuuuX-, (39.99, USD))
2067    (P

In [369]:
print(df['grade'].value_counts(dropna=False))
print(f"No grade (gradeless line): {df['grade'].isna().mean():.1%}")
no_grade = df[df['grade'].isna()]

grade
High Grade               1032
NaN                       433
Master Grade              299
30 Minutes Missions       122
Real Grade                115
30 Minutes Sisters        103
Super Deformed            101
Entry Grade                37
Perfect Grade              25
Advanced Grade             24
First Grade                11
Full Mechanics              9
Mega Size                   7
30 Minutes Preference       3
Name: count, dtype: int64
No grade (gradeless line): 18.7%


In [370]:
pd.set_option('display.max_rows', 100)
print(no_grade['classification'].value_counts())

classification
1/100 Gundam SEED Model Series                                                41
Figure-rise Standard                                                          31
1/144 Gundam SEED Model Series                                                27
1/100 Gundam 00 Model Series                                                  23
Mobile Suit Gundam Model Series                                               19
Reborn-One Hundred                                                            18
1/100 IRON-BLOODED ORPHANS                                                    16
1/144 Gundam SEED Model Series;1/144 Gundam SEED Destiny Collection Series    16
EX Model                                                                      13
1/144 G Gundam Model Series                                                   12
1/60 Gundam Model Series                                                      11
Haropla                                                                       11
Mobile Suit G

In [371]:
df = df.rename(columns={'model of': 'model_of'})
df['model_of'] = df['model_of'].apply(to_list)
df['kit_count'] = df['model_of'].apply(lambda x: max(len(x), 1) if isinstance(x, list) else pd.NA)

In [372]:
print(f"Missing price: {df['price_value'].isna().mean():.1%}")
print(f"Missing release_year: {df['release_year'].isna().mean():.1%}")
print(df['kit_count'].value_counts())

Missing price: 4.4%
Missing release_year: 15.3%
kit_count
1     1579
2      216
3       51
4       18
5        9
6        3
9        1
11       1
Name: count, dtype: int64


In [373]:
# Some wiki entries have the price under a different section, like isbn or need to paint (bruh)
MANUAL_PRICE_OVERRIDE = {
    "Figure-rise Standard Kamen Rider Kuuga (Mighty Form)/Decade Ver." : (3200, 'JPY'),
    "Super Mini-pla Shinka Gattai Daizyuzin": (4968, 'JPY')
}

for name, (value, currency) in MANUAL_PRICE_OVERRIDE.items():
    mask = df['kit_name'] == name
    df.loc[mask, 'price_value'] = value
    df.loc[mask, 'price_currency'] = currency

In [374]:
df.to_csv('Cleaned_Gunpla_Dataset.csv', index=False)